In [ ]:
# !pip install opencv-python

In [1]:
import os
import numpy as np
from scipy import ndimage
import cv2  # pip install opencv-python
import scipy.io # RETOUR A SCIPY ! Beaucoup plus simple.

# ==========================================
# 1. DÉFINITION DE VOS NOUVEAUX CHEMINS
# ==========================================
# On pointe directement vers le dossier IQ de InSilicoFlow !
DATA_PATH = "C:/Users/thoma/dev_projet/ecptr/PALA/PALA/PALA_data/PALA_data_InSilicoFlow/IQ"
OUTPUT_DIR = "dataset"

# Sous-dossiers requis par YOLO
DIRS = {
    "images_train": os.path.join(OUTPUT_DIR, "images/train"),
    "labels_train": os.path.join(OUTPUT_DIR, "labels/train"),
}

BOX_SIZE = 5

# ==========================================
# 2. FONCTION DE BRUITAGE
# ==========================================
def PALA_AddNoiseInIQ(IQ, power, impedance, sigmaGauss, clutterdB, amplCullerdB):
    max_IQ = np.max(IQ)
    noise = np.random.normal(size=np.prod(IQ.shape), scale=np.abs(power * impedance))
    noise = np.reshape(noise, IQ.shape)
    clutter = max_IQ * 10**(clutterdB / 20)
    ampl_clutter = max_IQ * 10**((amplCullerdB + clutterdB) / 20)
    return IQ + ndimage.gaussian_filter(clutter + noise * ampl_clutter, sigma=sigmaGauss)

# ==========================================
# 3. SCRIPT DE TEST (3 PREMIERS FICHIERS)
# ==========================================
def prepare_full_yolo_dataset():
    for path in DIRS.values():
        os.makedirs(path, exist_ok=True)
    
    NoiseParam = {"power": -2, "impedance": 0.2, "sigmaGauss": 1.5, "clutterdB": -20, "amplCullerdB": 10}
    
    print("Début de la génération du dataset complet...")
    
    # On boucle sur tous les fichiers de 1 à 20
    for i in range(1, 21):
        file_name = f"PALA_InSilicoFlow_IQ{i:03d}.mat"
        full_path = os.path.join(DATA_PATH, file_name)
        
        if not os.path.isfile(full_path):
            continue
            
        print(f"\n--- Traitement de {file_name} ---")
        mat_data = scipy.io.loadmat(full_path)
        
        iq_raw_full = np.abs(mat_data["IQ"])
        listpos_raw = mat_data["ListPos"]
        num_frames = listpos_raw.shape[2] # Devrait être 1000
        
        # On extrait 1 frame sur 10 pour avoir un dataset équilibré (100 images par fichier)
        for frame_idx in range(0, num_frames, 10):
            
            # --- 1. IMAGE ---
            iq_frame = iq_raw_full[:, :, frame_idx] if iq_raw_full.ndim == 3 else iq_raw_full
            noisy_iq = PALA_AddNoiseInIQ(iq_frame, **NoiseParam)
            # Sécurisation numérique
            noisy_iq = np.abs(noisy_iq)
            
            # Plancher minimal pour éviter log(0)
            eps = 1e-8
            noisy_iq = np.maximum(noisy_iq, eps)
            
            # Log compression
            log_iq = 20 * np.log10(noisy_iq)
            
            # Normalisation
            log_iq = np.nan_to_num(log_iq, nan=-120.0, neginf=-120.0)
            
            img_normalized = cv2.normalize(log_iq, None, 0, 255, cv2.NORM_MINMAX)
            img_png = np.uint8(img_normalized)
            h, w = img_png.shape
            
            img_filename = f"IQ{i:03d}_frame{frame_idx:04d}.png"
            cv2.imwrite(os.path.join(DIRS["images_train"], img_filename), img_png)
            
            # --- 2. COORDONNÉES ---
            txt_filename = f"IQ{i:03d}_frame{frame_idx:04d}.txt"
            txt_filepath = os.path.join(DIRS["labels_train"], txt_filename)
            
            with open(txt_filepath, 'w') as f_out:
                for bull_idx in range(listpos_raw.shape[0]):
                    x_pixel = listpos_raw[bull_idx, 0, frame_idx] # X de la frame courante
                    z_pixel = listpos_raw[bull_idx, 2, frame_idx] # Z de la frame courante
                    
                    if np.isnan(x_pixel) or np.isnan(z_pixel):
                        continue
                    
                    x_center = max(0.0, min(1.0, x_pixel / w))
                    y_center = max(0.0, min(1.0, z_pixel / h))
                    width_norm = BOX_SIZE / w
                    height_norm = BOX_SIZE / h
                    x_tl = x_center - width_norm
                    x_br = x_center + width_norm
                    y_tl = y_center - height_norm
                    y_br = y_center + height_norm
                    f_out.write(f"0 {x_tl:.6f} {y_tl:.6f} {x_br:.6f} {y_br:.6f}\n")

if __name__ == "__main__":
    prepare_full_yolo_dataset()

Début de la génération du dataset complet...

--- Traitement de PALA_InSilicoFlow_IQ001.mat ---

--- Traitement de PALA_InSilicoFlow_IQ002.mat ---

--- Traitement de PALA_InSilicoFlow_IQ003.mat ---

--- Traitement de PALA_InSilicoFlow_IQ004.mat ---

--- Traitement de PALA_InSilicoFlow_IQ005.mat ---

--- Traitement de PALA_InSilicoFlow_IQ006.mat ---

--- Traitement de PALA_InSilicoFlow_IQ007.mat ---

--- Traitement de PALA_InSilicoFlow_IQ008.mat ---

--- Traitement de PALA_InSilicoFlow_IQ009.mat ---

--- Traitement de PALA_InSilicoFlow_IQ010.mat ---

--- Traitement de PALA_InSilicoFlow_IQ011.mat ---

--- Traitement de PALA_InSilicoFlow_IQ012.mat ---

--- Traitement de PALA_InSilicoFlow_IQ013.mat ---

--- Traitement de PALA_InSilicoFlow_IQ014.mat ---

--- Traitement de PALA_InSilicoFlow_IQ015.mat ---

--- Traitement de PALA_InSilicoFlow_IQ016.mat ---

--- Traitement de PALA_InSilicoFlow_IQ017.mat ---

--- Traitement de PALA_InSilicoFlow_IQ018.mat ---

--- Traitement de PALA_InSilicoFlow_

In [21]:
import os
import numpy as np
from scipy import ndimage
import cv2
import scipy.io

# ==========================================
# 1. DÉFINITION DES CHEMINS
# ==========================================
DATA_PATH = "Données In Silico/PALA_data_InSilicoFlow/PALA_data_InSilicoFlow/IQ"
OUTPUT_DIR = "dataset"

DIRS = {
    "images_train": os.path.join(OUTPUT_DIR, "images/train"),
    "labels_train": os.path.join(OUTPUT_DIR, "labels/train"),
    "debug": os.path.join(OUTPUT_DIR, "debug") # Pour vérifier tes bounding boxes
}

BOX_SIZE = 5 # Taille de la box en pixels

# ==========================================
# 2. FONCTION DE BRUITAGE
# ==========================================
def PALA_AddNoiseInIQ(IQ, power, impedance, sigmaGauss, clutterdB, amplCullerdB):
    max_IQ = np.max(IQ)
    noise = np.random.normal(size=IQ.shape, scale=np.abs(power * impedance))
    clutter = max_IQ * 10**(clutterdB / 20)
    ampl_clutter = max_IQ * 10**((amplCullerdB + clutterdB) / 20)
    return IQ + ndimage.gaussian_filter(clutter + noise * ampl_clutter, sigma=sigmaGauss)

# ==========================================
# 3. GÉNÉRATION DU DATASET
# ==========================================
def prepare_full_yolo_dataset():
    for path in DIRS.values():
        os.makedirs(path, exist_ok=True)
    
    NoiseParam = {"power": -2, "impedance": 0.2, "sigmaGauss": 1.5, "clutterdB": -20, "amplCullerdB": 10}
    print("Début de la génération...")

    for i in range(1, 21):
        file_name = f"PALA_InSilicoFlow_IQ{i:03d}.mat"
        full_path = os.path.join(DATA_PATH, file_name)
        
        if not os.path.isfile(full_path):
            continue
            
        print(f"Traitement de {file_name}...")
        mat_data = scipy.io.loadmat(full_path)
        
        iq_raw_full = np.abs(mat_data["IQ"])
        listpos_raw = mat_data["ListPos"]
        num_frames = listpos_raw.shape[2]

        for frame_idx in range(0, num_frames, 10):
            # --- A. TRAITEMENT IMAGE ---
            iq_frame = iq_raw_full[:, :, frame_idx]
            noisy_iq = np.abs(PALA_AddNoiseInIQ(iq_frame, **NoiseParam))
            
            # Log compression et Normalisation
            log_iq = 20 * np.log10(np.maximum(noisy_iq, 1e-8))
            log_iq = np.nan_to_num(log_iq, nan=-120.0, neginf=-120.0)
            img_png = cv2.normalize(log_iq, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            h, w = img_png.shape
            
            base_name = f"IQ{i:03d}_frame{frame_idx:04d}"
            cv2.imwrite(os.path.join(DIRS["images_train"], f"{base_name}.png"), img_png)
            
            # --- B. LABELS & DEBUG ---
            img_debug = cv2.cvtColor(img_png, cv2.COLOR_GRAY2BGR)
            label_path = os.path.join(DIRS["labels_train"], f"{base_name}.txt")
            
            with open(label_path, 'w') as f_out:
                for bull_idx in range(listpos_raw.shape[0]):
                    x_pixel = listpos_raw[bull_idx, 0, frame_idx]
                    z_pixel = listpos_raw[bull_idx, 2, frame_idx] # Z correspond souvent à Y en image
                    
                    if np.isnan(x_pixel) or np.isnan(z_pixel):
                        continue

                    # YOLO Format: class x_center y_center width height (normalisé 0-1)
                    x_center_norm = x_pixel / w
                    y_center_norm = z_pixel / h
                    w_norm = BOX_SIZE / w
                    h_norm = BOX_SIZE / h
                    
                    f_out.write(f"0 {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                    
                    # Dessin Debug (en pixels)
                    x1 = int(x_pixel - BOX_SIZE/2)
                    y1 = int(z_pixel - BOX_SIZE/2)
                    x2 = int(x_pixel + BOX_SIZE/2)
                    y2 = int(z_pixel + BOX_SIZE/2)
                    cv2.rectangle(img_debug, (x1, y1), (x2, y2), (0, 0, 255), 1)

            cv2.imwrite(os.path.join(DIRS["debug"], f"{base_name}_debug.png"), img_debug)

if __name__ == "__main__":
    prepare_full_yolo_dataset()
    print("Terminé ! Coucou !")

Début de la génération...
Traitement de PALA_InSilicoFlow_IQ001.mat...
Traitement de PALA_InSilicoFlow_IQ002.mat...
Traitement de PALA_InSilicoFlow_IQ003.mat...
Traitement de PALA_InSilicoFlow_IQ004.mat...
Traitement de PALA_InSilicoFlow_IQ005.mat...
Traitement de PALA_InSilicoFlow_IQ006.mat...
Traitement de PALA_InSilicoFlow_IQ007.mat...
Traitement de PALA_InSilicoFlow_IQ008.mat...
Traitement de PALA_InSilicoFlow_IQ009.mat...
Traitement de PALA_InSilicoFlow_IQ010.mat...
Traitement de PALA_InSilicoFlow_IQ011.mat...
Traitement de PALA_InSilicoFlow_IQ012.mat...
Traitement de PALA_InSilicoFlow_IQ013.mat...
Traitement de PALA_InSilicoFlow_IQ014.mat...
Traitement de PALA_InSilicoFlow_IQ015.mat...
Traitement de PALA_InSilicoFlow_IQ016.mat...
Traitement de PALA_InSilicoFlow_IQ017.mat...
Traitement de PALA_InSilicoFlow_IQ018.mat...
Traitement de PALA_InSilicoFlow_IQ019.mat...
Traitement de PALA_InSilicoFlow_IQ020.mat...
Terminé ! Coucou !


In [23]:
import os
import numpy as np
from scipy import ndimage
import cv2
import scipy.io

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_PATH = "Données In Silico/PALA_data_InSilicoFlow/PALA_data_InSilicoFlow/IQ"
OUTPUT_DIR = "dataset"

DIRS = {
    "images_train": os.path.join(OUTPUT_DIR, "images/train"),
    "labels_train": os.path.join(OUTPUT_DIR, "labels/train"),
    "debug": os.path.join(OUTPUT_DIR, "debug")
}

BOX_SIZE_PIXELS = 7 # Taille de la boîte en pixels pour l'affichage/YOLO

def PALA_AddNoiseInIQ(IQ, power, impedance, sigmaGauss, clutterdB, amplCullerdB):
    max_IQ = np.max(IQ)
    noise = np.random.normal(size=IQ.shape, scale=np.abs(power * impedance))
    clutter = max_IQ * 10**(clutterdB / 20)
    ampl_clutter = max_IQ * 10**((amplCullerdB + clutterdB) / 20)
    return IQ + ndimage.gaussian_filter(clutter + noise * ampl_clutter, sigma=sigmaGauss)

# ==========================================
# 2. FONCTION DE CONVERSION MÈTRES -> PIXELS
# ==========================================
def meters_to_pixels(pos_m, axis_m):
    """
    Convertit une position en mètres vers un index pixel 
    en se basant sur le vecteur d'axe (P.x ou P.z).
    """
    # On trouve l'index le plus proche dans le vecteur d'axe
    return (np.abs(axis_m - pos_m)).argmin()

# ==========================================
# 3. GÉNÉRATION DU DATASET
# ==========================================
def prepare_full_yolo_dataset():
    for path in DIRS.values(): os.makedirs(path, exist_ok=True)
    
    NoiseParam = {"power": -2, "impedance": 0.2, "sigmaGauss": 1.5, "clutterdB": -20, "amplCullerdB": 10}
    
    for i in range(1, 21):
        file_name = f"PALA_InSilicoFlow_IQ{i:03d}.mat"
        full_path = os.path.join(DATA_PATH, file_name)
        if not os.path.isfile(full_path): continue
            
        print(f"Traitement de {file_name}...")
        mat_data = scipy.io.loadmat(full_path)
        
        # --- RÉCUPÉRATION DES AXES (Plus robuste) ---
        try:
            # Essai 1: Structure P standard (P['x'])
            x_axis = mat_data['P'][0,0]['x'].flatten()
            z_axis = mat_data['P'][0,0]['z'].flatten()
        except (ValueError, KeyError, IndexError):
            try:
                # Essai 2: Structure avec 'grid' (P['grid']['x'])
                grid = mat_data['P'][0,0]['grid'][0,0]
                x_axis = grid['x'].flatten()
                z_axis = grid['z'].flatten()
            except:
                # Essai 3: Variables à la racine du fichier .mat
                if 'x' in mat_data and 'z' in mat_data:
                    x_axis = mat_data['x'].flatten()
                    z_axis = mat_data['z'].flatten()
                else:
                    print(f"Erreur : Impossible de trouver les axes x et z dans {file_name}")
                    print("Champs dispos :", mat_data.keys())
                    continue
        
        iq_raw_full = np.abs(mat_data["IQ"])
        listpos_raw = mat_data["ListPos"]
        num_frames = listpos_raw.shape[2]

        for frame_idx in range(0, num_frames, 10):
            # --- IMAGE ---
            iq_frame = iq_raw_full[:, :, frame_idx]
            noisy_iq = np.abs(PALA_AddNoiseInIQ(iq_frame, **NoiseParam))
            log_iq = 20 * np.log10(np.maximum(noisy_iq, 1e-8))
            img_png = cv2.normalize(log_iq, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            h, w = img_png.shape
            
            base_name = f"IQ{i:03d}_frame{frame_idx:04d}"
            img_save_path = os.path.join(DIRS["images_train"], f"{base_name}.png")
            cv2.imwrite(img_save_path, img_png)
            
            # --- LABELS & DEBUG ---
            img_debug = cv2.cvtColor(img_png, cv2.COLOR_GRAY2BGR)
            label_path = os.path.join(DIRS["labels_train"], f"{base_name}.txt")
            
            with open(label_path, 'w') as f_out:
                for bull_idx in range(listpos_raw.shape[0]):
                    # Positions physiques (en mètres)
                    x_m = listpos_raw[bull_idx, 0, frame_idx]
                    z_m = listpos_raw[bull_idx, 2, frame_idx]
                    
                    if np.isnan(x_m) or np.isnan(z_m): continue

                    # CONVERSION : Mètres -> Index Pixel
                    x_px = meters_to_pixels(x_m, x_axis)
                    z_px = meters_to_pixels(z_m, z_axis)

                    # Normalisation pour YOLO (0 à 1)
                    x_center_norm = x_px / w
                    y_center_norm = z_px / h
                    w_norm = BOX_SIZE_PIXELS / w
                    h_norm = BOX_SIZE_PIXELS / h
                    
                    # Sécurité : on ignore si c'est hors image
                    if 0 <= x_center_norm <= 1 and 0 <= y_center_norm <= 1:
                        f_out.write(f"0 {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                        
                        # Dessin Debug
                        x1, y1 = int(x_px - BOX_SIZE_PIXELS/2), int(z_px - BOX_SIZE_PIXELS/2)
                        x2, y2 = int(x_px + BOX_SIZE_PIXELS/2), int(z_px + BOX_SIZE_PIXELS/2)
                        cv2.rectangle(img_debug, (x1, y1), (x2, y2), (0, 255, 0), 1) # Vert pour changer !

            cv2.imwrite(os.path.join(DIRS["debug"], f"{base_name}_debug.png"), img_debug)

if __name__ == "__main__":
    prepare_full_yolo_dataset()
    print("Fini ! Les boîtes devraient être centrées maintenant.")

Traitement de PALA_InSilicoFlow_IQ001.mat...
Erreur : Impossible de trouver les axes x et z dans PALA_InSilicoFlow_IQ001.mat
Champs dispos : dict_keys(['__header__', '__version__', '__globals__', 'Media', 'IQ', 'ListPos', 'P', 'PData', 'Resource', 'TW', 'TX', 'Trans'])
Traitement de PALA_InSilicoFlow_IQ002.mat...
Erreur : Impossible de trouver les axes x et z dans PALA_InSilicoFlow_IQ002.mat
Champs dispos : dict_keys(['__header__', '__version__', '__globals__', 'Media', 'IQ', 'ListPos', 'P', 'PData', 'Resource', 'TW', 'TX', 'Trans'])
Traitement de PALA_InSilicoFlow_IQ003.mat...
Erreur : Impossible de trouver les axes x et z dans PALA_InSilicoFlow_IQ003.mat
Champs dispos : dict_keys(['__header__', '__version__', '__globals__', 'Media', 'IQ', 'ListPos', 'P', 'PData', 'Resource', 'TW', 'TX', 'Trans'])
Traitement de PALA_InSilicoFlow_IQ004.mat...
Erreur : Impossible de trouver les axes x et z dans PALA_InSilicoFlow_IQ004.mat
Champs dispos : dict_keys(['__header__', '__version__', '__global

In [2]:
import os
import numpy as np
from scipy import ndimage
import cv2
import scipy.io

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_PATH = "C:/Users/thoma/dev_projet/ecptr/PALA/PALA/PALA_data/PALA_data_InSilicoFlow/IQ"
OUTPUT_DIR = "dataset"

DIRS = {
    "images_train": os.path.join(OUTPUT_DIR, "images/train"),
    "labels_train": os.path.join(OUTPUT_DIR, "labels/train"),
    "debug": os.path.join(OUTPUT_DIR, "debug")
}

BOX_SIZE_PIXELS = 7 

def PALA_AddNoiseInIQ(IQ, power, impedance, sigmaGauss, clutterdB, amplCullerdB):
    max_IQ = np.max(IQ)
    noise = np.random.normal(size=IQ.shape, scale=np.abs(power * impedance))
    clutter = max_IQ * 10**(clutterdB / 20)
    ampl_clutter = max_IQ * 10**((amplCullerdB + clutterdB) / 20)
    return IQ + ndimage.gaussian_filter(clutter + noise * ampl_clutter, sigma=sigmaGauss)

def meters_to_pixels(pos_m, axis_m):
    return (np.abs(axis_m - pos_m)).argmin()

# ==========================================
# 2. GÉNÉRATION DU DATASET
# ==========================================
def prepare_full_yolo_dataset():
    for path in DIRS.values(): os.makedirs(path, exist_ok=True)
    
    NoiseParam = {"power": -2, "impedance": 0.2, "sigmaGauss": 1.5, "clutterdB": -20, "amplCullerdB": 10}
    
    for i in range(1, 21):
        file_name = f"PALA_InSilicoFlow_IQ{i:03d}.mat"
        full_path = os.path.join(DATA_PATH, file_name)
        if not os.path.isfile(full_path): continue
            
        print(f"--- Traitement de {file_name} ---")
        mat_data = scipy.io.loadmat(full_path)
        
        iq_raw_full = np.abs(mat_data["IQ"])
        nz, nx = iq_raw_full.shape[0], iq_raw_full.shape[1]

        # --- RÉCUPÉRATION DES AXES VIA PDATA ---
        try:
            pdata = mat_data['PData'][0,0]
            # On récupère le pas (PDelta) et l'origine (Origin)
            # PDelta: [dx, dy, dz] | Origin: [x0, y0, z0]
            dx = pdata['PDelta'][0,0]
            dz = pdata['PDelta'][0,2]
            x0 = pdata['Origin'][0,0]
            z0 = pdata['Origin'][0,2]
            
            # Reconstruction des vecteurs d'axes
            x_axis = np.linspace(x0, x0 + (nx-1)*dx, nx)
            z_axis = np.linspace(z0, z0 + (nz-1)*dz, nz)
            
        except Exception as e:
            print(f"Erreur lors de l'extraction des axes dans {file_name}: {e}")
            continue

        listpos_raw = mat_data["ListPos"]
        num_frames = listpos_raw.shape[2]

        for frame_idx in range(0, num_frames, 10):
            # Traitement image
            iq_frame = iq_raw_full[:, :, frame_idx]
            noisy_iq = np.abs(PALA_AddNoiseInIQ(iq_frame, **NoiseParam))
            log_iq = 20 * np.log10(np.maximum(noisy_iq, 1e-8))
            img_png = cv2.normalize(log_iq, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            h, w = img_png.shape
            
            base_name = f"IQ{i:03d}_frame{frame_idx:04d}"
            cv2.imwrite(os.path.join(DIRS["images_train"], f"{base_name}.png"), img_png)
            
            img_debug = cv2.cvtColor(img_png, cv2.COLOR_GRAY2BGR)
            label_path = os.path.join(DIRS["labels_train"], f"{base_name}.txt")
            
            with open(label_path, 'w') as f_out:
                for bull_idx in range(listpos_raw.shape[0]):
                    x_m = listpos_raw[bull_idx, 0, frame_idx]
                    z_m = listpos_raw[bull_idx, 2, frame_idx]
                    
                    if np.isnan(x_m) or np.isnan(z_m): continue

                    # Conversion Mètres -> Pixels
                    x_px = meters_to_pixels(x_m, x_axis)
                    z_px = meters_to_pixels(z_m, z_axis)

                    # Normalisation YOLO
                    x_norm, y_norm = x_px / w, z_px / h
                    w_norm, h_norm = BOX_SIZE_PIXELS / w, BOX_SIZE_PIXELS / h
                    limit = BOX_SIZE_PIXELS//2
                    if 0 <= x_norm <= 1 and 0 <= y_norm <= 1:
                        f_out.write(f"0 {x_norm:.6f} {y_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                        # Debug visuel
                        cv2.rectangle(img_debug, (x_px-limit, z_px-limit), (x_px+limit, z_px+limit), (0, 255, 0), 1)

            cv2.imwrite(os.path.join(DIRS["debug"], f"{base_name}_debug.png"), img_debug)

if __name__ == "__main__":
    prepare_full_yolo_dataset()
    print("Terminé !")

--- Traitement de PALA_InSilicoFlow_IQ001.mat ---
--- Traitement de PALA_InSilicoFlow_IQ002.mat ---
--- Traitement de PALA_InSilicoFlow_IQ003.mat ---
--- Traitement de PALA_InSilicoFlow_IQ004.mat ---
--- Traitement de PALA_InSilicoFlow_IQ005.mat ---
--- Traitement de PALA_InSilicoFlow_IQ006.mat ---
--- Traitement de PALA_InSilicoFlow_IQ007.mat ---
--- Traitement de PALA_InSilicoFlow_IQ008.mat ---
--- Traitement de PALA_InSilicoFlow_IQ009.mat ---
--- Traitement de PALA_InSilicoFlow_IQ010.mat ---
--- Traitement de PALA_InSilicoFlow_IQ011.mat ---
--- Traitement de PALA_InSilicoFlow_IQ012.mat ---
--- Traitement de PALA_InSilicoFlow_IQ013.mat ---
--- Traitement de PALA_InSilicoFlow_IQ014.mat ---
--- Traitement de PALA_InSilicoFlow_IQ015.mat ---
--- Traitement de PALA_InSilicoFlow_IQ016.mat ---
--- Traitement de PALA_InSilicoFlow_IQ017.mat ---
--- Traitement de PALA_InSilicoFlow_IQ018.mat ---
--- Traitement de PALA_InSilicoFlow_IQ019.mat ---
--- Traitement de PALA_InSilicoFlow_IQ020.mat ---


In [3]:
import json
from pathlib import Path
from PIL import Image

def yolo_line_to_coco_bbox(line: str, img_w: int, img_h: int):
    """
    Convertit une ligne YOLO: class xc yc w h (normalisé 0..1)
    en bbox COCO: [x_min, y_min, width, height] en pixels.
    """
    parts = line.strip().split()
    if len(parts) != 5:
        raise ValueError(f"Ligne invalide (attendu 5 valeurs): {line!r}")

    cls, xc, yc, w, h = parts
    cls = int(cls)
    xc, yc, w, h = map(float, (xc, yc, w, h))

    # YOLO normalisé -> pixels
    xc *= img_w
    yc *= img_h
    w  *= img_w
    h  *= img_h

    x_min = xc - w / 2.0
    y_min = yc - h / 2.0

    # Clip dans l'image (sécurise les cas limites)
    x_min_clipped = max(0.0, min(x_min, img_w - 1.0))
    y_min_clipped = max(0.0, min(y_min, img_h - 1.0))

    # Ajuster w/h si le clip a coupé une partie
    x_max = min(x_min + w, img_w * 1.0)
    y_max = min(y_min + h, img_h * 1.0)
    w_clipped = max(0.0, x_max - x_min_clipped)
    h_clipped = max(0.0, y_max - y_min_clipped)

    bbox = [float(x_min_clipped), float(y_min_clipped), float(w_clipped), float(h_clipped)]
    area = float(w_clipped * h_clipped)

    return cls, bbox, area

def build_coco(images_dir: str, labels_dir: str, output_json: str):
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    output_json = Path(output_json)

    coco = {
        "info": {"description": "microbubbles dataset"},
        "licenses": [],
        "images": [],
        "categories": [
            {"id": 1, "name": "microbulle", "supercategory": "none"}
        ],
        "annotations": []
    }

    img_id = 1
    ann_id = 1

    # Parcourt les images (png)
    img_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() == ".png"])
    if not img_paths:
        raise FileNotFoundError(f"Aucune image .png trouvée dans: {images_dir}")

    for img_path in img_paths:
        with Image.open(img_path) as im:
            img_w, img_h = im.size

        coco["images"].append({
            "id": img_id,
            "file_name": img_path.name,
            "width": img_w,
            "height": img_h
        })

        label_path = labels_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            lines = [l.strip() for l in label_path.read_text().splitlines() if l.strip()]
            for line in lines:
                cls, bbox, area = yolo_line_to_coco_bbox(line, img_w, img_h)

                # Comme on ne garde qu'une classe, on mappe:
                # YOLO class 0 -> COCO category_id 1
                # (si tu as plusieurs classes plus tard, adapte ici)
                category_id = cls + 1

                # Ignore les bbox vides (peut arriver si w/h -> 0 après clip)
                if bbox[2] <= 0.0 or bbox[3] <= 0.0:
                    continue

                coco["annotations"].append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": category_id,
                    "bbox": bbox,
                    "area": area,
                    "iscrowd": 0,
                    "segmentation": []
                })
                ann_id += 1

        img_id += 1

    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(coco, ensure_ascii=False, indent=2))
    print(f"✅ COCO JSON écrit dans: {output_json}")
    print(f"Images: {len(coco['images'])} | Annotations: {len(coco['annotations'])} | Categories: {len(coco['categories'])}")


images = "dataset/images/train"
labels = "dataset/labels/train"
out = "result.json"

build_coco(images, labels, out)

✅ COCO JSON écrit dans: result
Images: 2000 | Annotations: 48844 | Categories: 1
